# Swept solid beam — v3: capture the torsional component of LTB

Exactly the same build + coupling as
`example_LTB_swept_solid_v2.ipynb` (sweep helper for the solid,
`node_to_surface_spring` for the end couplings, tet4 solid with
stiff elasticBeamColumn bridges). Two things change:

1. **Load target pushed to ~0.85 · M_cr** instead of 0.6. Perry
   amplification grows from `1/(1−0.6) ≈ 2.5×` to `1/(1−0.85) ≈
   6.7×` — enough for the coupled twist to grow from near-zero to
   a visible fraction of a degree at midspan.
2. **Midspan twist tracked directly from the solid mesh** instead of
   from a reference node rotation. The fork support locks `rx` at
   both ref points, so the twist at the ends is zero by BC — the
   interesting number is the twist at the midspan cross-section,
   which we extract by comparing the `uy` of two probe nodes at the
   top and bottom of the section.

### Twist from two nodal displacements

For a cross-section rotating about the beam axis by `θ_x` (small
rotations), a point at `(y, z)` in the section moves to
`(y - z·θ_x, z + y·θ_x)`. Picking two probe nodes at the same
midspan `x` and at `y ≈ 0` but different `z` (one near the top,
one near the bottom), the relative `uy` between them is

$$
u_y^{\text{bot}} - u_y^{\text{top}} \;=\; -(z_{\text{bot}} - z_{\text{top}})\,\theta_x
\;=\; +b_{\text{depth}}\,\theta_x
$$

so

$$
\theta_x^{\text{mid}} \;\approx\; \frac{u_y^{\text{bot}} - u_y^{\text{top}}}{z_{\text{bot}} - z_{\text{top}}}
$$

The rigid-body lateral translation component `u_y^{\text{centroid}}`
cancels out of this difference because both probe nodes carry the
same translation. What's left is the **pure twist**.

### Why this matters

LTB is a *coupled* mode: strong-axis bending drives lateral and
twist simultaneously. A lateral imperfection along the weak axis is
the conventional seed for the lowest LTB eigenshape — the twist
grows automatically via the geometric coupling in the TL tet4
kinematics. But the ratio is small: for a rectangular section,
`A_θ / A_y ≈ (1/L)·√(EI_weak/GJ)`, which for our geometry is about
`2·10⁻⁴ rad/mm` of lateral motion. So the twist only becomes
visible when the lateral amplification is big enough, which only
happens as we approach `M_cr`.

In [ ]:
from apeGmsh import apeGmsh, Results
import gmsh
import numpy as np
import matplotlib.pyplot as plt

# ---- Steel material ----------------------------------------------
E   = 200_000.0
nu  = 0.3
G   = E / (2.0 * (1 + nu))

# ---- Rectangular strip section -----------------------------------
b_depth = 200.0      # strong direction (z)
t_thk   = 20.0       # weak direction (y)

A        = b_depth * t_thk
I_strong = t_thk * b_depth ** 3 / 12.0
I_weak   = b_depth * t_thk ** 3 / 12.0
J        = (1.0 / 3.0) * b_depth * t_thk ** 3 * (1 - 0.63 * t_thk / b_depth)

# ---- Beam path + imperfection ------------------------------------
L       = 4000.0
delta_0 = L / 1000.0
lc      = 30.0

# ---- Stiff-beam section props for the spring links -------------
A_stiff = 1.0e6
I_stiff = 1.0e10
J_stiff = 1.0e10

# ---- Classical LTB reference moment (no warping) ----------------
M_cr_line = (np.pi / L) * np.sqrt(E * I_weak * G * J)

# ---- LTB eigenshape twist/lateral ratio -------------------------
# A_theta / A_y = (1/L) * sqrt(EI_weak / GJ) — for small sections,
# a small number: twist amplitude trails lateral amplitude.
ratio_theta_over_uy = (1.0 / L) * np.sqrt(E * I_weak / (G * J))

print(f'Rectangle {b_depth:.0f} x {t_thk:.0f} mm  '
      f'(I_strong/I_weak = {I_strong/I_weak:.0f})')
print(f'L = {L:.0f} mm    delta_0 = {delta_0:.2f} mm')
print(f'M_cr (no warping)  = {M_cr_line/1e6:.2f} kN*m')
print(f'LTB mode  \u03b8/u_y = {ratio_theta_over_uy:.4e} rad/mm')
print()
print(f'At 0.85 M_cr: Perry amplification = 1/0.15 = 6.67x')
print(f'  -> expected midspan u_y \u2248 6.67 \u00b7 {delta_0:.1f} = '
      f'{6.67*delta_0:.1f} mm')
print(f'  -> expected midspan twist    \u2248 '
      f'{6.67*delta_0*ratio_theta_over_uy*1e3:.2f} mrad '
      f'({np.degrees(6.67*delta_0*ratio_theta_over_uy):.2f} deg)')

## 1. Build the solid — same as v2

`add_line` → `replace_line(shape='sine', direction=(0, 1, 0))` →
rectangular profile → `m.model.geometry.sweep(...)` →
`m.constraints.node_to_surface_spring(...)`. No changes from v2.

In [ ]:
m = apeGmsh(model_name='swept_v3_twist', verbose=False)
m.begin()

# --- Imperfect path
p0 = m.model.geometry.add_point(0, 0, 0, lc=lc)
p1 = m.model.geometry.add_point(L, 0, 0, lc=lc)
path_line = m.model.geometry.add_line(p0, p1)
path_tags = m.model.geometry.replace_line(
    path_line,
    magnitude=delta_0,
    direction=(0, 1, 0),       # weak-axis lateral imperfection
    shape='sine',
    n_segments=16,
)

# --- Rectangular profile at x=0
corners = [
    (0, -t_thk/2, -b_depth/2),
    (0,  t_thk/2, -b_depth/2),
    (0,  t_thk/2,  b_depth/2),
    (0, -t_thk/2,  b_depth/2),
]
cp = [gmsh.model.occ.addPoint(*c) for c in corners]
cl = [gmsh.model.occ.addLine(cp[i], cp[(i + 1) % 4]) for i in range(4)]
prof_loop = gmsh.model.occ.addCurveLoop(cl)
prof_face = gmsh.model.occ.addPlaneSurface([prof_loop])
gmsh.model.occ.synchronize()

# --- Sweep
swept = m.model.geometry.sweep(prof_face, path_tags, label='beam_body')

# --- PG labels
gmsh.model.addPhysicalGroup(3, [swept['volume']], tag=-1, name='pg_beam')
gmsh.model.addPhysicalGroup(2, [swept['start_cap']], tag=-1, name='pin_face')
gmsh.model.addPhysicalGroup(2, [swept['end_cap']], tag=-1, name='roller_face')
gmsh.model.occ.synchronize()

# --- Reference points offset 50 mm outside each face
REF_OFFSET = 50.0
m.model.geometry.add_point(-REF_OFFSET,    0, 0, lc=lc, label='ref_pin')
m.model.geometry.add_point(L + REF_OFFSET, 0, 0, lc=lc, label='ref_roller')

# --- Spring-variant couplings
m.constraints.node_to_surface_spring('ref_pin',    'pin_face')
m.constraints.node_to_surface_spring('ref_roller', 'roller_face')

# --- Mesh
m.mesh.sizing.set_global_size(lc)
m.mesh.generation.generate(dim=3)
fem = m.mesh.queries.get_fem_data(remove_orphans=True)
m.end()

print(f'nodes: {len(fem.nodes.ids)}')
for g in fem.elements:
    print(f'  {g.type_name:6s} n={len(g)}')

## 2. Locate the twist probe nodes

Pick two mesh nodes on the midspan cross-section that bracket the
section depth vertically and lie on the beam centreline in the
weak-axis direction: one **near the top flange** at `z ≈ +b/2,
y ≈ 0, x ≈ L/2` and one **near the bottom flange** at `z ≈ -b/2,
y ≈ 0, x ≈ L/2`.

The `uy` difference between these two nodes, divided by their `z`
separation, is the midspan twist angle.

In [ ]:
beam_ids    = np.array(
    [int(n) for n in fem.nodes.get_ids(target='pg_beam')])
beam_coords = fem.nodes.get_coords(target='pg_beam')

# Nearest node to (L/2, 0, +b/2) and (L/2, 0, -b/2)
target_top = np.array([L/2, 0.0, +b_depth/2])
target_bot = np.array([L/2, 0.0, -b_depth/2])

top_idx = int(np.argmin(np.linalg.norm(beam_coords - target_top, axis=1)))
bot_idx = int(np.argmin(np.linalg.norm(beam_coords - target_bot, axis=1)))

top_nid = int(beam_ids[top_idx])
bot_nid = int(beam_ids[bot_idx])
top_xyz = beam_coords[top_idx]
bot_xyz = beam_coords[bot_idx]
dz_probe = float(top_xyz[2] - bot_xyz[2])

# Also a plain centroid midspan node for lateral/vertical tracking
mid_idx = int(np.argmin(
    np.linalg.norm(beam_coords - np.array([L/2, 0, 0]), axis=1)))
mid_nid = int(beam_ids[mid_idx])
mid_xyz = beam_coords[mid_idx]

print(f'midspan centroid probe: node {mid_nid} @ '
      f'({mid_xyz[0]:.0f}, {mid_xyz[1]:+.1f}, {mid_xyz[2]:+.1f})')
print(f'top    twist probe:     node {top_nid} @ '
      f'({top_xyz[0]:.0f}, {top_xyz[1]:+.1f}, {top_xyz[2]:+.1f})')
print(f'bottom twist probe:     node {bot_nid} @ '
      f'({bot_xyz[0]:.0f}, {bot_xyz[1]:+.1f}, {bot_xyz[2]:+.1f})')
print(f'probe dz = {dz_probe:+.2f} mm')

## 3. Emit the OpenSees model

Exactly the same emission as v2: tet4 solid at `ndf=3`, ref points +
phantoms at `ndf=6`, stiff `elasticBeamColumn` bridges for the
`node_to_surface_spring` couplings, `equalDOF` for phantom →
solid-face, fork support BCs at the ref points.

In [ ]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)
ops.timeSeries('Linear', 1)

for nid, xyz in fem.nodes.get(target='pg_beam'):
    ops.node(int(nid), *xyz, '-ndf', 3)
for nid, xyz in fem.nodes.get(target=['ref_pin', 'ref_roller']):
    ops.node(int(nid), *xyz)
for nid, xyz in fem.nodes.constraints.phantom_nodes():
    ops.node(int(nid), *xyz)

ops.nDMaterial('ElasticIsotropic', 1, E, nu)
max_eid = 0
for group in fem.elements.get(element_type='tet4'):
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', int(eid), *conn, 1)
        max_eid = max(max_eid, int(eid))

ops.geomTransf(
    'Linear', 1,
    1.0 / np.sqrt(3), 1.0 / np.sqrt(3), 1.0 / np.sqrt(3),
)
next_eid = max_eid + 1
for master, slaves in fem.nodes.constraints.stiff_beam_groups():
    for slave in slaves:
        ops.element(
            'elasticBeamColumn', next_eid,
            int(master), int(slave),
            A_stiff, E, G, J_stiff, I_stiff, I_stiff, 1,
        )
        next_eid += 1
for pair in fem.nodes.constraints.equal_dofs():
    ops.equalDOF(pair.master_node, pair.slave_node, *pair.dofs)

ref_pin_id  = int(fem.nodes.get_ids(target='ref_pin')[0])
ref_roll_id = int(fem.nodes.get_ids(target='ref_roller')[0])
ops.fix(ref_pin_id,  1, 1, 1, 1, 0, 0)
ops.fix(ref_roll_id, 1, 1, 1, 1, 0, 0)

ops.pattern('Plain', 1, 1)
ops.load(ref_pin_id,  0, 0, 0, 0, +1.0, 0)
ops.load(ref_roll_id, 0, 0, 0, 0, -1.0, 0)

print('OpenSees model built')
print(f'  tet4 elements + stiff beams + equalDOF')

## 4. Sweep to 0.85 M_cr with twist tracking

`KrylovNewton` with `NormDispIncr 1e-5` tolerance, 40 load-control
steps to 0.85 M_cr_line. At each step we record:

* **`M`** — applied moment
* **`u_y(mid)`** — lateral midspan centroid displacement
* **`u_z(mid)`** — vertical midspan centroid displacement
* **`θ_x(mid)`** — twist, computed from `(u_y^{bot} − u_y^{top}) / dz`
  on the two flange-centreline probe nodes

In [ ]:
ops.constraints('Transformation')
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-5, 300)
ops.algorithm('KrylovNewton')

M_target = 0.85 * M_cr_line
n_steps  = 40
dM       = M_target / n_steps
ops.integrator('LoadControl', dM)
ops.analysis('Static')

emitted_ids = set(int(n) for n in beam_ids)
emitted_ids.add(ref_pin_id)
emitted_ids.add(ref_roll_id)
for pid in fem.nodes.constraints.phantom_nodes().ids:
    emitted_ids.add(int(pid))
id_to_row = {int(nid): i for i, nid in enumerate(fem.nodes.ids)}

hist_M     = []
hist_midy  = []
hist_midz  = []
hist_twist = []
disp_per_step: list[np.ndarray] = []
n_nodes = len(fem.nodes.ids)

print(f'{"step":>5s} {"M [kN*m]":>10s} {"M/M_cr":>8s} '
      f'{"uy [mm]":>10s} {"uz [mm]":>10s} {"twist [mrad]":>14s}')

for step in range(n_steps):
    ok = ops.analyze(1)
    if ok != 0:
        print(f'failed at step {step + 1}, ok={ok}')
        break
    M_now = (step + 1) * dM

    d_mid = ops.nodeDisp(mid_nid)
    d_top = ops.nodeDisp(top_nid)
    d_bot = ops.nodeDisp(bot_nid)

    twist = (d_bot[1] - d_top[1]) / dz_probe

    hist_M.append(M_now)
    hist_midy.append(d_mid[1])
    hist_midz.append(d_mid[2])
    hist_twist.append(twist)

    if (step + 1) % 5 == 0 or step == 0:
        print(f'{step+1:5d} {M_now/1e6:10.3f} '
              f'{M_now/M_cr_line*100:7.1f}%  '
              f'{d_mid[1]:+10.4f} {d_mid[2]:+10.4f} '
              f'{twist*1e3:+14.4f}')

    d_full = np.zeros((n_nodes, 3), dtype=np.float64)
    for nid in emitted_ids:
        if nid not in id_to_row:
            continue
        di = ops.nodeDisp(int(nid))
        row = id_to_row[nid]
        d_full[row, 0] = di[0]
        d_full[row, 1] = di[1]
        d_full[row, 2] = di[2]
    disp_per_step.append(d_full)

if not hist_M:
    raise RuntimeError('analysis did not converge on any step')

print()
print(f'converged in {len(hist_M)} of {n_steps} steps')
print(f'final M          = {hist_M[-1]/1e6:.3f} kN*m  '
      f'({hist_M[-1]/M_cr_line*100:.1f}% M_cr_line)')
print(f'final mid u_y    = {hist_midy[-1]:+.4f} mm')
print(f'final mid u_z    = {hist_midz[-1]:+.4f} mm')
print(f'final mid twist  = {hist_twist[-1]:+.4e} rad  '
      f'({np.degrees(hist_twist[-1]):+.3f} deg)')

# Linearity check: ratio of twist/load at the first and last steps.
# If the coupling is alive and Perry-amplifying, twist should grow
# faster than linearly with load — the ratio at the end should be
# larger than the ratio at the beginning.
ratio_first = hist_twist[0] / hist_M[0]
ratio_last  = hist_twist[-1] / hist_M[-1]
print(f'\ntwist/M ratio: first step={ratio_first:+.3e}, '
      f'last step={ratio_last:+.3e}')
print(f'nonlinearity factor (last/first) = {ratio_last/ratio_first:.2f}')

## 5. Load–deformation curves and the honest result\n\nFour panels — strong-axis vertical, weak-axis lateral, midspan twist, and the twist-vs-lateral coupling.\n\n**Important note on what the FEM actually shows.** Step-by-step output\nfrom the previous cell reveals that every tracked quantity (`u_y`, `u_z`,\n`twist`) grows **perfectly linearly** with the applied moment. The\n`twist/M` ratio at step 40 is numerically identical to the ratio at step\n1 — there is **no Perry amplification**. This is not a bug in the\nworkflow; it's a consequence of OpenSees's `FourNodeTetrahedron`\nformulation. The element computes infinitesimal strains from the\nreference (initial) configuration and does not include a geometric\nstiffness contribution, so `K` never updates with the deformed state.\nNewton on a linear-elastic material + small-strain `K` is just a\nsingle-step linear solve in disguise. LTB is a bifurcation that\ndepends on the full finite-rotation `K + K_g` — something this\nelement does not carry.\n\nWhat you *do* see is **genuine linear coupling**: the imperfect\ngeometry (centroid line offset by `δ₀·sin(π·x/L)`) means that\nstrong-axis bending produces a small but non-zero twist via 3-D\ncontinuum equilibrium. The ratio `θ_x / u_y` matches the LTB\neigenmode ratio\n\n$$\n\\frac{A_\\theta}{A_y} \\;\\approx\\; \\frac{1}{L}\\sqrt{\\frac{E I_{\\text{weak}}}{G J}}\n$$\n\nto within geometric noise, and the twist at 85 % of `M_cr_line` is\n~0.05° at midspan — present, measurable, and correctly captured by\nthe coupled 3-D kinematics. Just not amplified.\n\n### When does this matter, and when doesn't it?\n\nThe swept-solid + `node_to_surface_spring` workflow is still exactly\nright for:\n\n- Visualising the imperfect geometry in 3-D\n- Checking local stresses, cross-section flange / web effects,\n  support reactions, and other quantities that line elements cannot\n  represent\n- Linear pre-buckling analysis on a solid body with clean BC control\n\nIt is **not the right tool** for extracting a quantitative LTB limit\nload. For that, use the **line-element corotational approach** from\n`example_LTB_uniform_moment.ipynb` — `elasticBeamColumn` with\n`geomTransf('Corotational', ...)` does carry `K_g`, and it\nreproduces Perry–Robertson to within 1% on the same beam. The line\nand solid tools complement each other: the line gives you the\nphysics number, the solid gives you the local detail.

In [ ]:
hist_M     = np.asarray(hist_M)
hist_midy  = np.asarray(hist_midy)
hist_midz  = np.asarray(hist_midz)
hist_twist = np.asarray(hist_twist)

# Perry envelope for the lateral component
eta = hist_M / M_cr_line
valid = eta < 0.99
uy_env = delta_0 / (1.0 - eta[valid])

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

# (0,0) Strong-axis vertical deflection
ax = axes[0, 0]
ax.plot(hist_midz, hist_M / 1e6, 'o-', lw=1.4, ms=3, color='tab:blue')
ax.set_xlabel('midspan u_z [mm]')
ax.set_ylabel('M  [kN*m]')
ax.set_title('Strong-axis bending (vertical)')
ax.grid(True, alpha=0.3)

# (0,1) Weak-axis lateral + Perry envelope
ax = axes[0, 1]
ax.plot(delta_0 + hist_midy, hist_M / 1e6, 'o-', lw=1.4, ms=3,
        color='tab:red', label='FEM (solid)')
ax.plot(uy_env, hist_M[valid] / 1e6, 'k--', lw=1.2,
        label='Perry  \u03b4\u2080/(1-M/M_cr)')
ax.axvline(delta_0, color='tab:gray', lw=0.8, ls=':',
           label=f'\u03b4\u2080 = {delta_0:.2f} mm')
ax.axhline(M_cr_line / 1e6, color='k', lw=0.8, ls=':',
           label=f'M_cr = {M_cr_line/1e6:.1f} kN*m')
ax.set_xlabel('midspan total lateral [mm]')
ax.set_ylabel('M  [kN*m]')
ax.set_title('Weak-axis amplification')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=8)

# (1,0) Twist vs M
ax = axes[1, 0]
ax.plot(np.degrees(hist_twist), hist_M / 1e6, 'o-', lw=1.4, ms=3,
        color='tab:purple')
ax.axhline(M_cr_line / 1e6, color='k', lw=0.8, ls=':')
ax.set_xlabel('midspan twist \u03b8_x [deg]')
ax.set_ylabel('M  [kN*m]')
ax.set_title('Torsional component of LTB')
ax.grid(True, alpha=0.3)

# (1,1) Twist as a function of lateral (coupling check)
ax = axes[1, 1]
lateral_total = delta_0 + hist_midy
ax.plot(lateral_total, np.degrees(hist_twist), 'o-', lw=1.4, ms=3,
        color='tab:green')
# Analytical LTB mode shape ratio: theta / u_y = ratio_theta_over_uy
u_y_line = np.array([lateral_total.min(), lateral_total.max()])
theta_line = np.degrees(u_y_line * ratio_theta_over_uy)
ax.plot(u_y_line, theta_line, 'k--', lw=1.0,
        label=f'LTB eigenmode ratio')
ax.set_xlabel('midspan lateral u_y [mm]')
ax.set_ylabel('midspan twist \u03b8_x [deg]')
ax.set_title('Twist vs. lateral (coupled mode shape)')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Time-series viewer

Time axis = applied moment `M` in N·mm. Scrubbing through the load
history shows the imperfect solid tilting out of its initial plane
— the cross-section at midspan visibly rotates about the beam axis
as `M` approaches `M_cr`.

In [ ]:
steps = []
for M_i, d_step in zip(hist_M, disp_per_step):
    u_mag = np.linalg.norm(d_step, axis=1)
    steps.append({
        'time': float(M_i),
        'point_data': {
            'Displacement': d_step,
            '|U|':          u_mag,
            'Uy':           d_step[:, 1],
            'Uz':           d_step[:, 2],
        },
    })

print(f'time-series steps: {len(steps)}')
print(f'time range       : {steps[0]["time"]/1e6:.2f} to '
      f'{steps[-1]["time"]/1e6:.2f} kN*m')

results = Results.from_fem(
    fem,
    steps=steps,
    include_pgs=['pg_beam'],
    name='swept_solid_v3_twist',
)
results.viewer(blocking=False)